# UT2 · 1 — NumPy en profundidad

Este cuaderno es el primero de los cuatro cuadernos prácticos de la Unidad 2 (NumPy · Pandas · Matplotlib · Reto integrador). Aquí nos centramos solo en **NumPy**: crear arrays, indexarlos, combinarlos y aprovechar la vectorización.

No hace falta ningún archivo externo — todo se genera dentro del propio cuaderno. Cada bloque tiene primero una celda de práctica guiada y, en varios bloques, un mini-reto o un ejercicio para intentarlo antes de mirar la solución.

In [1]:
import numpy as np

np.random.seed(42)
print("NumPy cargado correctamente. Versión:", np.__version__)

NumPy cargado correctamente. Versión: 2.4.4


## 1 · De listas a arrays, y los tipos de dato (dtype)

Un array de NumPy se parece a una lista de Python, pero por dentro es muy distinto: todos sus elementos son del **mismo tipo** y viven en un bloque de memoria contiguo. Eso es justo lo que le permite ser tan rápido.

In [2]:
precios_lista = [10, 20, 30]
precios_array = np.array(precios_lista)

print("Lista   * 2:", precios_lista * 2)   # repite la secuencia
print("Array   * 2:", precios_array * 2)   # multiplica cada elemento

print("\nTipo de la lista:", type(precios_lista))
print("Tipo del array:   ", type(precios_array))
print("dtype del array:  ", precios_array.dtype)

Lista   * 2: [10, 20, 30, 10, 20, 30]
Array   * 2: [20 40 60]

Tipo de la lista: <class 'list'>
Tipo del array:    <class 'numpy.ndarray'>
dtype del array:   int64


### El dtype se ajusta automáticamente ("upcasting")

Si mezclas enteros y decimales, NumPy sube todo el array al tipo más "amplio" para no perder información — no se queda a medias.

In [3]:
solo_enteros = np.array([1, 2, 3])
con_decimal = np.array([1, 2, 3.0])          # un solo float "contamina" a todo el array
forzado_texto = np.array([1, 2, 3], dtype=str)

print("Solo enteros:  ", solo_enteros.dtype)
print("Con un decimal:", con_decimal.dtype)
print("Forzado a str: ", forzado_texto.dtype, forzado_texto)

# astype convierte explícitamente, sin depender de que NumPy lo intuya
enteros_desde_float = con_decimal.astype(int)
print("astype(int):   ", enteros_desde_float, enteros_desde_float.dtype)

Solo enteros:   int64
Con un decimal: float64
Forzado a str:  <U1 ['1' '2' '3']
astype(int):    [1 2 3] int64


### Cuidado: mezclar tipos muy distintos no siempre da lo que esperas

Cuando mezclas números con texto, o booleanos con números, NumPy no lanza un error — simplemente "sube" todo al tipo más general posible. Esto puede sorprenderte si no lo esperas.

In [4]:
lista_mixta = [1, "dos", 3.0, True]
array_mixto = np.array(lista_mixta)

print(array_mixto)
print("dtype:", array_mixto.dtype)    # todo se convierte a texto (string) — ya no puedes sumar estos elementos como números

['1' 'dos' '3.0' 'True']
dtype: <U32


**Ejercicio:** antes de ejecutar la siguiente celda, predice qué `dtype` tendrá el array resultante. Luego ejecútala y comprueba si acertabas. Pista: `True` y `False` se comportan como `1` y `0` para NumPy.

In [5]:
prediccion = np.array([True, False, 1, 0, 7])
print(prediccion, "· dtype:", prediccion.dtype)
# booleano + entero -> todo sube a entero (int), porque un entero puede representar tanto 0/1 como cualquier otro número

[1 0 1 0 7] · dtype: int64


## 2 · Indexado y slicing: 1D, 2D y máscaras booleanas

In [6]:
m = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print("Primera fila:       ", m[0])
print("Última fila:        ", m[-1])
print("Segunda columna:    ", m[:, 1])
print("Sub-matriz 2x2:      \n", m[0:2, 0:2])
print("Elemento fila 2, col 1:", m[2, 1])
print("Elementos > 5:       ", m[m > 5])          # máscara booleana
print("Forma de la matriz:  ", m.shape)

Primera fila:        [1 2 3]
Última fila:         [7 8 9]
Segunda columna:     [2 5 8]
Sub-matriz 2x2:      
 [[1 2]
 [4 5]]
Elemento fila 2, col 1: 8
Elementos > 5:        [6 7 8 9]
Forma de la matriz:   (3, 3)


### Slicing con paso, e indexado "fancy" (con listas de índices)

In [7]:
puntuaciones = np.array([3, 7, 5, 9, 2, 6, 8, 4, 10, 1])

print("De 2 en 2:              ", puntuaciones[::2])
print("Al revés:               ", puntuaciones[::-1])
print("Últimos 3:              ", puntuaciones[-3:])
print("Posiciones concretas:   ", puntuaciones[[0, 3, 7]])   # fancy indexing: una lista como índice

altas_mask = puntuaciones >= 5
print("\n¿Puntuación alta?      ", altas_mask)
print("Puntuaciones altas:     ", puntuaciones[altas_mask])

De 2 en 2:               [ 3  5  2  8 10]
Al revés:                [ 1 10  4  8  6  2  9  5  7  3]
Últimos 3:               [ 4 10  1]
Posiciones concretas:    [3 9 4]

¿Puntuación alta?       [False  True  True  True False  True  True False  True False]
Puntuaciones altas:      [ 7  5  9  6  8 10]


**Reto rápido:** de `puntuaciones`, obtén en una sola línea el array con las tres notas más bajas, ordenadas de menor a mayor. Pista: combina `np.sort` con un slice.

In [8]:
# Solución del reto
tres_mas_bajas = np.sort(puntuaciones)[:3]
print("Las tres puntuaciones más bajas:", tres_mas_bajas)

Las tres puntuaciones más bajas: [1 2 3]


## 3 · np.where: selección y modificación condicional

In [9]:
# np.where(condición, valor_si_true, valor_si_false) es el "si/si no" vectorizado
etiquetas = np.where(puntuaciones >= 6, "alta", "baja")
print("Etiquetas:", etiquetas)

# también sirve para "recortar" valores por debajo de un mínimo, sin bucles
recortadas = np.where(puntuaciones < 4, 4, puntuaciones)
print("Recortadas (mínimo 4):", recortadas)

# y para encontrar POSICIONES en vez de valores, np.where con una sola condición devuelve los índices
posiciones_altas = np.where(puntuaciones >= 8)
print("Posiciones con puntuación >= 8:", posiciones_altas[0])

Etiquetas: ['baja' 'alta' 'baja' 'alta' 'baja' 'alta' 'alta' 'baja' 'alta' 'baja']
Recortadas (mínimo 4): [ 4  7  5  9  4  6  8  4 10  4]
Posiciones con puntuación >= 8: [3 6 8]


## 4 · Broadcasting: operar arrays de formas distintas

NumPy no obliga a que los dos arrays de una operación tengan exactamente la misma forma: si las formas son "compatibles", "repite" mentalmente el más pequeño para que encajen, sin duplicar datos en memoria.

In [10]:
tabla = np.array([[1, 2, 3], [4, 5, 6]])          # 2 filas x 3 columnas
factor_por_columna = np.array([10, 100, 1000])      # 1 fila x 3 columnas

print("tabla * factor_por_columna:\n", tabla * factor_por_columna)
# NumPy "repite" mentalmente factor_por_columna en cada fila de tabla

tabla * factor_por_columna:
 [[  10  200 3000]
 [  40  500 6000]]


### Broadcasting también funciona "por filas" con un vector columna

In [11]:
factor_por_fila = np.array([[1], [2]])              # 2 filas x 1 columna

print("tabla * factor_por_fila:\n", tabla * factor_por_fila)
# aquí se repite el vector a lo largo de las columnas, no de las filas

# Regla rápida: dos dimensiones son compatibles si son iguales, o si una de ellas es 1
print("\nForma de tabla:            ", tabla.shape)
print("Forma de factor_por_columna:", factor_por_columna.shape)
print("Forma de factor_por_fila:   ", factor_por_fila.shape)

tabla * factor_por_fila:
 [[ 1  2  3]
 [ 8 10 12]]

Forma de tabla:             (2, 3)
Forma de factor_por_columna: (3,)
Forma de factor_por_fila:    (2, 1)


## 5 · Funciones para crear arrays

No siempre partimos de una lista: muchas veces necesitamos crear un array "desde cero" con una forma concreta.

In [12]:
np.random.seed(42)   # fija la semilla: mismos "aleatorios" cada vez que se ejecuta

ceros = np.zeros((2, 3))
unos = np.ones((2, 3))
llenos = np.full((2, 3), 7)
identidad = np.eye(3)
diagonal = np.diag([1, 2, 3])
secuencia = np.arange(0, 10, 2)          # de 0 a 10 (sin incluir), de 2 en 2
espaciado = np.linspace(0, 1, 5)          # 5 valores igualmente repartidos entre 0 y 1

print("zeros:\n", ceros)
print("full:\n", llenos)
print("eye:\n", identidad)
print("diag:\n", diagonal)
print("arange:", secuencia)
print("linspace:", espaciado)

zeros:
 [[0. 0. 0.]
 [0. 0. 0.]]
full:
 [[7 7 7]
 [7 7 7]]
eye:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
diag:
 [[1 0 0]
 [0 2 0]
 [0 0 3]]
arange: [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]


### Números aleatorios: uniforme, normal y elección entre opciones

In [13]:
aleatorios_uniformes = np.random.random((2, 3))                   # entre 0 y 1
aleatorios_enteros = np.random.randint(0, 10, size=(2, 3))         # enteros entre 0 y 9
aleatorios_normales = np.random.normal(loc=6, scale=2, size=5)     # distribución normal, media 6, desv. 2
eleccion = np.random.choice(["PC", "Consola", "Móvil"], size=6)    # elige al azar entre opciones dadas

print("random uniforme:\n", np.round(aleatorios_uniformes, 2))
print("randint:\n", aleatorios_enteros)
print("normal (redondeado):", np.round(aleatorios_normales, 2))
print("choice:", eleccion)

random uniforme:
 [[0.37 0.95 0.73]
 [0.6  0.16 0.16]]
randint:
 [[7 4 3]
 [7 7 2]]
normal (redondeado): [5.06 7.09 5.07 5.07 6.48]
choice: ['PC' 'PC' 'Móvil' 'Móvil' 'Móvil' 'Consola']


### Ejercicio aplicado: generar datos de Instagram

Imagina que quieres simular los "me gusta" de las últimas 10 publicaciones de una cuenta, para probar una gráfica antes de tener datos reales. Genera un array de 10 enteros aleatorios entre 50 y 5000.

In [14]:
likes_ultimas_10_publicaciones = np.random.randint(50, 5000, size=10)
print("Likes:", likes_ultimas_10_publicaciones)
print("Total acumulado:", likes_ultimas_10_publicaciones.sum())
print("Publicación con más likes (posición):", likes_ultimas_10_publicaciones.argmax())

Likes: [2097 2797 1025 1856  239 2784 3055 4708 1949 1317]
Total acumulado: 21827
Publicación con más likes (posición): 7


## 6 · Inspeccionar, reorganizar, combinar y partir arrays

In [15]:
print("shape:", m.shape, " ndim:", m.ndim, " size:", m.size, " dtype:", m.dtype)

# reshape cambia la forma sin cambiar los datos
plano = m.reshape(9)
print("reshape a 1D:", plano)
print("traspuesta:\n", m.T)

# combinar arrays
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print("concatenate:", np.concatenate([a, b]))
print("vstack:\n", np.vstack([a, b]))
print("hstack:", np.hstack([a, b]))

shape: (3, 3)  ndim: 2  size: 9  dtype: int64
reshape a 1D: [1 2 3 4 5 6 7 8 9]
traspuesta:
 [[1 4 7]
 [2 5 8]
 [3 6 9]]
concatenate: [1 2 3 4 5 6]
vstack:
 [[1 2 3]
 [4 5 6]]
hstack: [1 2 3 4 5 6]


### Combinar matrices por eje, y partir un array en trozos

In [16]:
fila_extra = np.array([[10, 11, 12]])
print("concatenate axis=0 (añade una fila):\n", np.concatenate([m, fila_extra], axis=0))

columna_extra = np.array([[10], [11], [12]])
print("\nconcatenate axis=1 (añade una columna):\n", np.concatenate([m, columna_extra], axis=1))

# np.split hace lo contrario: partir un array en trozos iguales
mitades = np.split(np.arange(10), 2)
print("\nnp.split en 2 mitades:", mitades)

concatenate axis=0 (añade una fila):
 [[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]

concatenate axis=1 (añade una columna):
 [[ 1  2  3 10]
 [ 4  5  6 11]
 [ 7  8  9 12]]

np.split en 2 mitades: [array([0, 1, 2, 3, 4]), array([5, 6, 7, 8, 9])]


### reshape vs. resize: parecidos, pero no son lo mismo

Los dos cambian la "forma" de un array, pero con una diferencia importante: `reshape` exige que el número total de elementos no cambie y te devuelve casi siempre una **vista** (no toca el array original salvo que reasignes); `resize` modifica el array **en el sitio** y, si le pides más elementos de los que hay, rellena los huecos con ceros.

In [17]:
a = np.arange(6)

b = a.reshape(2, 3)          # mismo nº de elementos (6 = 2x3), no modifica a
print("reshape (2,3):\n", b)
print("¿a ha cambiado?", a)

c = a.copy()
c.resize(2, 4)               # 8 "huecos" para solo 6 elementos -> rellena con 0, y SÍ modifica c en el sitio
print("\nresize (2,4), rellena con ceros:\n", c)

reshape (2,3):
 [[0 1 2]
 [3 4 5]]
¿a ha cambiado? [0 1 2 3 4 5]

resize (2,4), rellena con ceros:
 [[0 1 2 3]
 [4 5 0 0]]


## 7 · Vistas y copias: la trampa más habitual de NumPy

Al hacer un slice de un array, NumPy no crea datos nuevos por defecto — te da una **vista** que sigue apuntando a la misma memoria. Si no lo sabes, puedes modificar sin querer el array original.

In [18]:
original = np.array([1, 2, 3, 4, 5])
vista = original[1:4]        # esto NO copia los datos, solo "recorta la ventana"

vista[0] = 999
print("Array original tras modificar la vista:", original)   # ¡también cambió!

# Para evitarlo, hay que pedir una copia explícita con .copy()
copia = original[1:4].copy()
copia[0] = -1
print("Array original tras modificar la copia:  ", original)   # esta vez no cambia

Array original tras modificar la vista: [  1 999   3   4   5]
Array original tras modificar la copia:   [  1 999   3   4   5]


**Reto rápido:** ¿el resultado de `original.reshape(...)` es una vista o una copia? Compruébalo tú: crea un array, cambia su forma con `reshape`, modifica un elemento del resultado y observa si el array original cambia también.

In [19]:
# Solución del reto
prueba = np.arange(6)
prueba_reshape = prueba.reshape(2, 3)
prueba_reshape[0, 0] = -100
print("Array original:", prueba)   # cambia también -> reshape() normalmente devuelve una vista, no una copia

Array original: [-100    1    2    3    4    5]


## 8 · Agregados por eje: axis=0 vs. axis=1

Cuando trabajas con matrices, casi toda función de agregación (`sum`, `mean`, `max`...) acepta un parámetro `axis` para decidir si resume por filas o por columnas.

In [20]:
horas_semana = np.array([
    [2, 3, 1, 4],   # jugador A: horas jugadas cada semana
    [5, 4, 6, 5],   # jugador B
    [1, 2, 2, 1],   # jugador C
])

print("Total por semana (axis=0, colapsa filas):   ", horas_semana.sum(axis=0))
print("Total por jugador (axis=1, colapsa columnas):", horas_semana.sum(axis=1))
print("Media semanal por jugador:                   ", horas_semana.mean(axis=1))
print("Jugador con más horas en cada semana (argmax):", horas_semana.argmax(axis=0))

Total por semana (axis=0, colapsa filas):    [ 8  9  9 10]
Total por jugador (axis=1, colapsa columnas): [10 20  6]
Media semanal por jugador:                    [2.5 5.  1.5]
Jugador con más horas en cada semana (argmax): [1 1 1 1]


### Más estadísticos: mediana y varianza

La media se deja influir mucho por valores extremos; la mediana no. Y la varianza es la "materia prima" de la que sale la desviación típica (`std` es la raíz cuadrada de `var`).

In [21]:
print("Media:            ", puntuaciones.mean())
print("Mediana:          ", np.median(puntuaciones))
print("Varianza:         ", round(puntuaciones.var(), 2))
print("Desviación típica:", round(puntuaciones.std(), 2), " (= raíz cuadrada de la varianza)")
print("Máximo / mínimo:  ", puntuaciones.max(), "/", puntuaciones.min())

Media:             5.5
Mediana:           5.5
Varianza:          8.25
Desviación típica: 2.87  (= raíz cuadrada de la varianza)
Máximo / mínimo:   10 / 1


## 9 · Ordenar, buscar y contar valores

In [22]:
print("Ordenado (menor a mayor):", np.sort(puntuaciones))
print("Ordenado (mayor a menor):  ", np.sort(puntuaciones)[::-1])

orden = np.argsort(puntuaciones)          # índices que ordenarían el array
print("Índices de orden:", orden)
print("Puntuaciones según ese orden:", puntuaciones[orden])

Ordenado (menor a mayor): [ 1  2  3  4  5  6  7  8  9 10]
Ordenado (mayor a menor):   [10  9  8  7  6  5  4  3  2  1]


Índices de orden: [9 4 0 7 2 5 1 6 3 8]
Puntuaciones según ese orden: [ 1  2  3  4  5  6  7  8  9 10]


### np.unique: valores distintos y cuántas veces aparece cada uno

In [23]:
plataforma_de_cada_partida = np.array(["PC", "Consola", "PC", "PC", "Móvil", "Consola", "PC"])

valores_distintos, conteos = np.unique(plataforma_de_cada_partida, return_counts=True)
print("Plataformas distintas:", valores_distintos)
print("Cuántas veces aparece cada una:", conteos)

for plataforma, veces in zip(valores_distintos, conteos):
    print(f"  {plataforma}: {veces} partidas")

Plataformas distintas: ['Consola' 'Móvil' 'PC']
Cuántas veces aparece cada una: [2 1 4]
  Consola: 2 partidas
  Móvil: 1 partidas
  PC: 4 partidas


**Ejercicio:** tienes el array `series_vistas` con el nombre de la serie que cada compañero de clase dice que está viendo ahora mismo. Sin mirar la solución, averigua cuál es la serie más repetida (la "moda") usando `np.unique` con `return_counts=True`.

In [24]:
# Solución del ejercicio
series_vistas = np.array(["Wednesday", "The Bear", "Wednesday", "Bridgerton", "The Bear", "Wednesday", "House of the Dragon"])

nombres, veces = np.unique(series_vistas, return_counts=True)
serie_mas_vista = nombres[np.argmax(veces)]
print("Serie más mencionada:", serie_mas_vista, f"({veces.max()} veces)")

Serie más mencionada: Wednesday (3 veces)


## 10 · ¿Se mueven juntas dos variables? np.corrcoef

Antes de construir cualquier modelo, es habitual comprobar si dos variables numéricas están relacionadas: cuando una sube, ¿la otra también sube (o baja)?

In [25]:
horas_de_esos_jugadores = np.array([1, 5, 3, 9, 0, 4, 7, 2])   # mismo nº de elementos que puntuaciones (8, no 10)

matriz_correlacion = np.corrcoef(puntuaciones[:8], horas_de_esos_jugadores)
print("Matriz de correlación:\n", np.round(matriz_correlacion, 2))
print("\nCorrelación entre ambas variables:", round(matriz_correlacion[0, 1], 2))
# cerca de 1: suben juntas · cerca de -1: una sube cuando la otra baja · cerca de 0: no hay relación lineal clara

Matriz de correlación:
 [[1.   0.99]
 [0.99 1.  ]]

Correlación entre ambas variables: 0.99


## 11 · Valores que faltan: NaN en NumPy

No solo las tablas de Pandas tienen huecos — un array numérico también puede tenerlos, representados con `np.nan` ("Not a Number"). Esto es justo el mismo problema que vamos a resolver con Pandas en el próximo cuaderno, pero empezamos a verlo aquí porque conviene saber cómo se comporta a nivel de NumPy.

In [26]:
visualizaciones_capitulo = np.array([120000, 98000, np.nan, 105000, np.nan, 87000, 99000])  # algún día falló el contador

print("¿Dónde hay huecos?   ", np.isnan(visualizaciones_capitulo))
print("Nº de huecos:        ", np.isnan(visualizaciones_capitulo).sum())

# la media "normal" se contagia de NaN: si un solo elemento es NaN, el resultado también lo es
print("\nMedia ingenua (sale NaN):", visualizaciones_capitulo.mean())
print("Media ignorando huecos:  ", np.nanmean(visualizaciones_capitulo))   # versión que sí funciona

rellenado = np.nan_to_num(visualizaciones_capitulo, nan=np.nanmean(visualizaciones_capitulo))
print("\nArray con huecos rellenados con la media:", np.round(rellenado))

¿Dónde hay huecos?    [False False  True False  True False False]
Nº de huecos:         2

Media ingenua (sale NaN): nan
Media ignorando huecos:   101800.0

Array con huecos rellenados con la media: [120000.  98000. 101800. 105000. 101800.  87000.  99000.]


## 12 · Guardar y cargar arrays

Cuando un array tarda en calcularse (o viene de un experimento largo), conviene guardarlo en disco en vez de recalcularlo cada vez.

In [27]:
np.save("puntuaciones_backup.npy", puntuaciones)          # formato binario propio de NumPy
puntuaciones_recuperadas = np.load("puntuaciones_backup.npy")
print("¿Son iguales tras guardar y cargar?", np.array_equal(puntuaciones, puntuaciones_recuperadas))

# también se puede guardar como texto plano, más lento pero legible/editable a mano
np.savetxt("puntuaciones_backup.csv", puntuaciones, delimiter=",", fmt="%d")
desde_csv = np.loadtxt("puntuaciones_backup.csv", delimiter=",")
print("Recuperado desde CSV:", desde_csv)

¿Son iguales tras guardar y cargar? True
Recuperado desde CSV: [ 3.  7.  5.  9.  2.  6.  8.  4. 10.  1.]


## 13 · Experimento: bucle Python vs. vectorización

La razón práctica de todo esto: un array de NumPy no solo es más cómodo de escribir, es mucho más rápido que un bucle `for` equivalente.

In [28]:
import time

n = 1_000_000
datos = np.random.random(n)

inicio = time.time()
resultado_bucle = [x * 2 for x in datos]
t_bucle = time.time() - inicio

inicio = time.time()
resultado_vector = datos * 2
t_vector = time.time() - inicio

print(f"Bucle Python:  {t_bucle:.4f} s")
print(f"Vectorizado:   {t_vector:.4f} s")
print(f"Vectorizado es aprox. {t_bucle / t_vector:.0f}x más rápido en esta máquina")

Bucle Python:  1.0391 s
Vectorizado:   0.1675 s
Vectorizado es aprox. 6x más rápido en esta máquina


## 14 · Reto integrador de NumPy

Con lo que llevas practicado en este cuaderno: dado el array `partidas` de abajo (puntuación obtenida en 12 partidas), calcula...

1. la media, la mediana y la desviación típica,
2. cuántas partidas superan la media,
3. las 3 mejores partidas (valores, no posiciones), ordenadas de mayor a menor,
4. una versión "normalizada" del array donde cada valor está entre 0 y 1 (pista: `(x - min) / (max - min)`).

In [29]:
partidas = np.array([12, 45, 8, 33, 27, 41, 19, 5, 38, 22, 30, 15])

media = partidas.mean()
mediana = np.median(partidas)
desviacion = partidas.std()
por_encima_de_la_media = (partidas > media).sum()
top_3 = np.sort(partidas)[::-1][:3]
normalizado = (partidas - partidas.min()) / (partidas.max() - partidas.min())

print("Media:                    ", round(media, 2))
print("Mediana:                  ", mediana)
print("Desviación típica:        ", round(desviacion, 2))
print("Partidas sobre la media:  ", por_encima_de_la_media)
print("Top 3 mejores partidas:   ", top_3)
print("Normalizado (0 a 1):      ", np.round(normalizado, 2))

Media:                     24.58
Mediana:                   24.5
Desviación típica:         12.65
Partidas sobre la media:   6
Top 3 mejores partidas:    [45 41 38]
Normalizado (0 a 1):       [0.18 1.   0.08 0.7  0.55 0.9  0.35 0.   0.82 0.42 0.62 0.25]


---
**Resumen de lo practicado:** creación de arrays y dtypes (incluyendo mezclas de tipos) · indexado, slicing y fancy indexing · `np.where` · broadcasting · funciones de creación (`zeros`, `arange`, `linspace`, `random`...) · reshape/resize, concatenar y partir arrays · vistas vs. copias · agregados por eje, mediana y varianza · ordenar, `np.unique` y correlación · NaN en NumPy · guardar/cargar arrays.

**Siguiente cuaderno:** `UT2_02_pandas.ipynb`, donde estos mismos arrays se convierten en las columnas de un DataFrame — y donde los huecos (NaN) por fin tienen herramientas cómodas para limpiarlos.